# AI 신용평가 모형 (Credit Scoring with ML + LLM)

**목표**: 개인의 금융 데이터를 기반으로 대출 연체 여부를 예측하는 AI 모형 구축

| 단계 | 내용 |
|------|------|
| 1 | 데이터 탐색 (EDA) |
| 2 | 전처리 |
| 3 | 모형 학습 (LightGBM) |
| 4 | 성능 평가 |
| 5 | LLM 해석 (선택) |

---
## 0. 라이브러리 불러오기

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
import pickle

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

# 한글 폰트 설정 (없으면 무시됨)
try:
    plt.rcParams['font.family'] = 'NanumGothic'
    plt.rcParams['axes.unicode_minus'] = False
except:
    pass

# 출력 폴더
OUTPUT_DIR = os.path.join('..', 'outputs')
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Ready!')

---
## 1. 데이터 불러오기 & 탐색 (EDA)

In [ ]:
# 데이터 불러오기
df = pd.read_csv(os.path.join('..', 'data', 'credit_data.csv'))

print(f'데이터 크기: {df.shape[0]:,}명, {df.shape[1]}개 변수')
print(f'연체율: {df["default"].mean():.1%}')
print()
df.head()

In [ ]:
# 기초 통계량
df.describe().round(2)

In [ ]:
# 결측치 확인
missing = df.isnull().sum()
print('결측치 현황:')
print(missing[missing > 0] if missing.sum() > 0 else '결측치 없음!')

In [ ]:
# Target 분포: 연체 vs 정상
fig, ax = plt.subplots(1, 1, figsize=(6, 4))
counts = df['default'].value_counts()
colors = ['#2ecc71', '#e74c3c']
bars = ax.bar(['Normal (0)', 'Default (1)'], counts.values, color=colors, edgecolor='white')

for bar, count in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 50,
            f'{count:,} ({count/len(df):.1%})', ha='center', fontweight='bold')

ax.set_title('Target Distribution: Default vs Normal', fontsize=14, fontweight='bold')
ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'target_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 주요 변수 분포 시각화
features_to_plot = ['age', 'monthly_income', 'debt_ratio', 'revolving_utilization',
                     'times_90_plus_days_late', 'num_open_credit_lines']

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, col in enumerate(features_to_plot):
    ax = axes[i]
    for label, color in [(0, '#2ecc71'), (1, '#e74c3c')]:
        subset = df[df['default'] == label][col]
        ax.hist(subset, bins=30, alpha=0.6, color=color,
                label=f'Default={label}', density=True)
    ax.set_title(col, fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)

plt.suptitle('Feature Distributions by Default Status', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'feature_distributions.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 상관관계 히트맵
fig, ax = plt.subplots(figsize=(10, 8))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=ax, square=True)
ax.set_title('Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'correlation_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## 2. 데이터 전처리

In [ ]:
from sklearn.model_selection import train_test_split

# Feature / Target 분리
X = df.drop('default', axis=1)
y = df['default']

print(f'Feature 수: {X.shape[1]}')
print(f'Feature 목록: {list(X.columns)}')

In [ ]:
# 학습 / 테스트 분할 (80% : 20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'학습 데이터: {X_train.shape[0]:,}명')
print(f'테스트 데이터: {X_test.shape[0]:,}명')
print(f'학습 연체율: {y_train.mean():.1%}')
print(f'테스트 연체율: {y_test.mean():.1%}')

---
## 3. 모형 학습 (LightGBM)

**LightGBM**: 결정 트리를 순차적으로 학습하여 이전 트리의 실수를 보완하는 Gradient Boosting 모형.  
정형 데이터에서 가장 강력한 모형 중 하나.

In [ ]:
import lightgbm as lgb

# LightGBM 데이터셋 생성
train_data = lgb.Dataset(X_train, label=y_train)
valid_data = lgb.Dataset(X_test, label=y_test, reference=train_data)

# 하이퍼파라미터 설정
params = {
    'objective': 'binary',          # 이진 분류
    'metric': 'auc',                # AUC로 평가
    'learning_rate': 0.05,          # 학습률 (작을수록 정밀, 느림)
    'num_leaves': 31,               # 트리 복잡도
    'max_depth': 6,                 # 트리 깊이 제한
    'min_child_samples': 20,        # 리프 최소 샘플 수
    'feature_fraction': 0.8,        # 변수 샘플링 비율
    'bagging_fraction': 0.8,        # 데이터 샘플링 비율
    'bagging_freq': 5,
    'is_unbalance': True,           # 불균형 데이터 처리
    'verbose': -1,                  # 로그 숨기기
    'seed': 42,
}

print('하이퍼파라미터:')
for k, v in params.items():
    print(f'  {k}: {v}')

In [ ]:
# 모형 학습
print('모형 학습 시작...\n')

callbacks = [lgb.log_evaluation(period=50)]

model = lgb.train(
    params,
    train_data,
    num_boost_round=300,            # 트리 300개 학습
    valid_sets=[train_data, valid_data],
    valid_names=['train', 'valid'],
    callbacks=callbacks,
)

print(f'\n학습 완료! Best iteration: {model.best_iteration}')

In [ ]:
# 모형 저장
model_path = os.path.join(OUTPUT_DIR, 'model.pkl')
with open(model_path, 'wb') as f:
    pickle.dump(model, f)
print(f'모형 저장: {model_path}')

---
## 4. 모형 평가

In [ ]:
from sklearn.metrics import (
    roc_auc_score, roc_curve,
    confusion_matrix, classification_report,
    precision_recall_curve, average_precision_score
)

# 예측 확률
y_pred_proba = model.predict(X_test)
y_pred = (y_pred_proba >= 0.5).astype(int)

# AUC Score
auc_score = roc_auc_score(y_test, y_pred_proba)
print(f'=== 모형 성능 ===' )
print(f'AUC-ROC Score: {auc_score:.4f}')
print()
print(classification_report(y_test, y_pred, target_names=['Normal', 'Default']))

In [ ]:
# ROC Curve 시각화
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)

fig, ax = plt.subplots(figsize=(7, 7))
ax.plot(fpr, tpr, color='#3498db', lw=2.5, label=f'LightGBM (AUC = {auc_score:.4f})')
ax.plot([0, 1], [0, 1], color='gray', lw=1, linestyle='--', label='Random (AUC = 0.5000)')
ax.fill_between(fpr, tpr, alpha=0.1, color='#3498db')
ax.set_xlabel('False Positive Rate', fontsize=13)
ax.set_ylabel('True Positive Rate', fontsize=13)
ax.set_title('ROC Curve', fontsize=15, fontweight='bold')
ax.legend(fontsize=12, loc='lower right')
ax.set_xlim([-0.01, 1.01])
ax.set_ylim([-0.01, 1.01])
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'roc_curve.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 혼동행렬 (Confusion Matrix)
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Normal', 'Default'],
            yticklabels=['Normal', 'Default'],
            annot_kws={'size': 16})
ax.set_xlabel('Predicted', fontsize=13)
ax.set_ylabel('Actual', fontsize=13)
ax.set_title('Confusion Matrix', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'confusion_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'True Negative (정상->정상):  {tn:,}')
print(f'False Positive (정상->연체): {fp:,}  <- 정상인데 연체로 예측 (Type I Error)')
print(f'False Negative (연체->정상): {fn:,}  <- 연체인데 정상으로 예측 (Type II Error, 위험!)')
print(f'True Positive (연체->연체):  {tp:,}')

In [ ]:
# 변수 중요도 (Feature Importance)
importance = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importance(importance_type='gain')
}).sort_values('importance', ascending=True)

fig, ax = plt.subplots(figsize=(8, 6))
bars = ax.barh(importance['feature'], importance['importance'], color='#3498db', edgecolor='white')

# 가장 중요한 변수 강조
bars[-1].set_color('#e74c3c')
bars[-2].set_color('#e67e22')
bars[-3].set_color('#f39c12')

ax.set_xlabel('Importance (Gain)', fontsize=13)
ax.set_title('Feature Importance', fontsize=15, fontweight='bold')
ax.grid(True, axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'feature_importance.png'), dpi=150, bbox_inches='tight')
plt.show()

print('\n=== Top 5 Important Features ===')
for _, row in importance.tail(5).iloc[::-1].iterrows():
    print(f"  {row['feature']}: {row['importance']:.0f}")

---
## 5. 개별 고객 예측 & 해석

In [ ]:
def predict_and_explain(model, X_test, y_test, idx=0):
    """개별 고객의 예측 결과를 보기 좋게 출력"""
    customer = X_test.iloc[idx]
    actual = y_test.iloc[idx]
    prob = model.predict(customer.values.reshape(1, -1))[0]

    risk_level = 'HIGH RISK' if prob >= 0.5 else ('MEDIUM' if prob >= 0.3 else 'LOW RISK')
    risk_color = '\033[91m' if prob >= 0.5 else ('\033[93m' if prob >= 0.3 else '\033[92m')
    reset = '\033[0m'

    print('=' * 50)
    print(f'  Customer #{idx}')
    print('=' * 50)
    print(f'  Default Probability: {risk_color}{prob:.1%}{reset}')
    print(f'  Risk Level:          {risk_color}{risk_level}{reset}')
    print(f'  Actual:              {"DEFAULT" if actual == 1 else "NORMAL"}')
    print('-' * 50)
    print('  Profile:')
    print(f'    Age:                {customer["age"]:.0f}')
    print(f'    Monthly Income:     ${customer["monthly_income"]:,.0f}')
    print(f'    Debt Ratio:         {customer["debt_ratio"]:.2f}')
    print(f'    Revolving Util:     {customer["revolving_utilization"]:.1%}')
    print(f'    90+ Days Late:      {customer["times_90_plus_days_late"]:.0f} times')
    print(f'    60-89 Days Late:    {customer["times_60_89_days_late"]:.0f} times')
    print(f'    30-59 Days Late:    {customer["times_30_59_days_late"]:.0f} times')
    print(f'    Open Credit Lines:  {customer["num_open_credit_lines"]:.0f}')
    print(f'    Real Estate Loans:  {customer["num_real_estate_loans"]:.0f}')
    print(f'    Dependents:         {customer["num_dependents"]:.0f}')
    print('=' * 50)

    return customer, prob

# 고위험 고객 예시
high_risk_idx = y_pred_proba.argmax()
print('>>> High Risk Customer Example')
customer_hr, prob_hr = predict_and_explain(model, X_test, y_test, high_risk_idx)
print()

# 저위험 고객 예시
low_risk_idx = y_pred_proba.argmin()
print('>>> Low Risk Customer Example')
customer_lr, prob_lr = predict_and_explain(model, X_test, y_test, low_risk_idx)

---
## 6. LLM 해석 (Claude / ChatGPT)

아래 프롬프트를 Claude 또는 ChatGPT에 복사-붙여넣기하면 AI가 신용평가 결과를 해석해줍니다.  
(API 키가 있으면 코드로 자동화도 가능)

In [ ]:
def generate_llm_prompt(customer, prob, feature_importance_top5):
    """LLM에 넣을 프롬프트 생성 (복사-붙여넣기용)"""

    prompt = f"""당신은 은행의 신용평가 전문가입니다.
AI 신용평가 모형의 결과를 바탕으로 고객의 신용 상태를 분석하고,
대출 심사 의견을 작성해주세요.

=== AI 모형 예측 결과 ===
연체 확률: {prob:.1%}
위험 등급: {'고위험' if prob >= 0.5 else ('중위험' if prob >= 0.3 else '저위험')}

=== 고객 정보 ===
- 나이: {customer['age']:.0f}세
- 월 소득: {customer['monthly_income']:,.0f}원
- 부채 비율: {customer['debt_ratio']:.2f} (월소득 대비)
- 신용카드 사용률: {customer['revolving_utilization']:.1%}
- 90일 이상 연체: {customer['times_90_plus_days_late']:.0f}회
- 60-89일 연체: {customer['times_60_89_days_late']:.0f}회
- 30-59일 연체: {customer['times_30_59_days_late']:.0f}회
- 신용 계좌 수: {customer['num_open_credit_lines']:.0f}개
- 부동산 대출: {customer['num_real_estate_loans']:.0f}건
- 부양가족: {customer['num_dependents']:.0f}명

=== 모형이 가장 중요하게 본 변수 (Top 5) ===
{feature_importance_top5}

위 정보를 바탕으로 다음을 작성해주세요:
1. 종합 신용 평가 (2-3문장)
2. 주요 위험/강점 요인 (각 2-3개)
3. 대출 심사 의견 (승인/조건부승인/거절 + 근거)
4. 개선 권고사항 (고객에게 제안할 사항)
"""
    return prompt

# Top 5 변수 중요도 문자열
top5_str = '\n'.join([
    f"  {i+1}. {row['feature']}: 중요도 {row['importance']:.0f}"
    for i, (_, row) in enumerate(importance.tail(5).iloc[::-1].iterrows())
])

# 고위험 고객 프롬프트
prompt = generate_llm_prompt(customer_hr, prob_hr, top5_str)

print('=' * 60)
print('  아래 프롬프트를 Claude 또는 ChatGPT에 붙여넣기하세요')
print('=' * 60)
print(prompt)

### (선택) Claude API 자동 호출

API 키가 있다면 아래 코드로 자동 해석이 가능합니다.  
없으면 위 프롬프트를 웹에서 직접 사용하세요.

In [ ]:
# [선택] Claude API 자동 호출
# 사용하려면: pip install anthropic
# 그리고 ANTHROPIC_API_KEY 환경변수 설정 필요

USE_API = False  # True로 바꾸면 API 호출

if USE_API:
    try:
        import anthropic
        client = anthropic.Anthropic()  # ANTHROPIC_API_KEY 환경변수에서 자동 로드

        message = client.messages.create(
            model='claude-sonnet-4-20250514',
            max_tokens=1024,
            messages=[{'role': 'user', 'content': prompt}]
        )

        print('=== Claude AI 신용평가 해석 ===')
        print(message.content[0].text)
    except ImportError:
        print('anthropic 패키지가 설치되지 않았습니다.')
        print('pip install anthropic')
    except Exception as e:
        print(f'API 호출 오류: {e}')
else:
    print('API 모드 비활성. 위의 프롬프트를 Claude/ChatGPT 웹에서 사용하세요.')

---
## 7. 결과 요약

### 모형 성능 요약

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

results = {
    'AUC-ROC': roc_auc_score(y_test, y_pred_proba),
    'Precision': precision_score(y_test, y_pred),
    'Recall': recall_score(y_test, y_pred),
    'F1 Score': f1_score(y_test, y_pred),
}

print('=' * 40)
print('  Final Model Performance')
print('=' * 40)
for metric, value in results.items():
    bar = '#' * int(value * 30)
    print(f'  {metric:12s}: {value:.4f}  {bar}')
print('=' * 40)
print()
print('Saved outputs:')
for f in os.listdir(OUTPUT_DIR):
    fpath = os.path.join(OUTPUT_DIR, f)
    size = os.path.getsize(fpath)
    print(f'  {f} ({size/1024:.1f} KB)')

---

## Next Steps

1. **다른 모형 비교**: `from sklearn.ensemble import RandomForestClassifier` 로 Random Forest와 비교
2. **SHAP 해석**: `pip install shap` 후 개별 예측의 변수별 기여도 분석
3. **실제 데이터**: Kaggle 'Give Me Some Credit' 데이터로 교체
4. **웹 데모**: `pip install streamlit` 후 인터랙티브 대시보드 구축
5. **RAG 연동**: 금융 규제 문서를 검색하여 LLM 해석에 근거 추가